# Minggu 12 — Praktik: Menentukan Lokasi Gempa

**Seismologi PAGF262413** · Program Studi Sarjana Geofisika FMIPA UGM

Data: **enam gempa susulan Yogyakarta 2006**, waktu tempuh P dan S di 17 stasiun jaringan YK.

| Kelompok | Event | Gap azimut | Artinya |
|:--|:--|:--|:--|
| Geometri **baik** | EV01–EV03 | 53–59° | Jaringan mengepung gempa |
| Geometri **buruk** | EV04–EV06 | 188–272° | Jaringan hanya melihat dari satu sisi |

**Inti tugas ini adalah membandingkan keduanya.** Kalian akan menemukan sendiri bahwa metode yang sama, dengan data sama banyaknya, memberi hasil yang jauh berbeda ketelitiannya.

---

### Catatan penting tentang data

Kolom `P` dan `S` berisi **waktu tempuh** dari waktu asal, bukan waktu tiba jam dinding. Jadi waktu asal $t_0 = 0$ menurut konstruksi. Kalau garis Wadati kalian tidak memotong tepat di nol, **itu bukan kesalahan** — itu ukuran seberapa jauh anggapan medium seragam menyimpang dari bumi berlapis yang sebenarnya.

**AI boleh dipakai sebebasnya.** Yang dinilai: galat episenter kalian terhadap katalog, mutu penalaran di sel bertanda ✍️, dan **kejujuran selang ketidakpastian** kalian.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
plt.rcParams['figure.figsize']=(7,6); plt.rcParams['axes.grid']=True; plt.rcParams['grid.alpha']=.25

tt   = pd.read_csv('data/W12_waktu_tiba.csv')
sta  = pd.read_csv('data/W12_stasiun.csv').set_index('sta')
kat  = pd.read_csv('data/W12_katalog_rujukan.csv').set_index('event')
KM_PER_DEG = 111.19
print(kat.round(3).to_string())

## Bagian 1 — Diagram Wadati

Plot $(S-P)$ terhadap $P$ untuk semua stasiun yang punya kedua fase. Dari kemiringannya kalian
memperoleh $V_P/V_S$ — sifat elastik kerak di bawah Bantul, dari sebelas selisih waktu.

In [ ]:
EVENT = 'EV01'                 # <-- ganti sesuai penugasan
NAMA  = 'tulis nama kalian'    # <-- wajib

d = tt[tt.event == EVENT].dropna(subset=['P','S']).copy()
d['SP'] = d.S - d.P
print(f"{EVENT}: {len(d)} stasiun berpasangan P-S | gap katalog {kat.loc[EVENT,'gap']:.0f}°")

slope, inter = np.polyfit(d.P, d.SP, 1)
vpvs = 1 + slope
poisson = ...                  # <-- TUGAS: hitung rasio Poisson dari vpvs

plt.plot(d.P, d.SP, 'o')
xs = np.linspace(0, d.P.max()*1.1, 10); plt.plot(xs, slope*xs + inter, '--')
for _, r in d.iterrows(): plt.annotate(r.sta, (r.P, r.SP), textcoords='offset points', xytext=(6,-3), fontsize=8)
plt.xlabel('Waktu tempuh P (s)'); plt.ylabel('S - P (s)')
plt.title(f'Wadati {EVENT} — Vp/Vs = {vpvs:.3f}')
print(f'Vp/Vs = {vpvs:.3f} | perpotongan = {-inter/slope:+.3f} s')

✍️ **Jawaban 1:**

1. Berapa $V_P/V_S$ dan rasio Poisson kalian? Apakah nilainya wajar untuk batuan kerak?
2. Perpotongannya berapa? Ingat $t_0$ sejatinya nol — apa arti selisihnya?

*(tulis di sini)*

## Bagian 2 — Metode lingkaran

Ubah $(S-P)$ menjadi jarak **hiposentral**, lalu proyeksikan ke peta:

$$R_{\text{peta}} = \sqrt{d_{\text{hiposentral}}^2 - h^2}$$

Coba beberapa nilai $h$. Perhatikan pada nilai berapa lingkarannya paling memusat.

In [ ]:
from matplotlib.patches import Circle
VP = 5.5                       # km/s, anggapan awal
k  = VP/slope                  # faktor pengubah (S-P) -> jarak hiposentral
d['dhyp'] = d.SP * k

H = 10.0                       # <-- TUGAS: coba-coba nilai kedalaman ini

fig, ax = plt.subplots(figsize=(7.5,7))
for _, r in d.iterrows():
    if r.sta not in sta.index: continue
    R2 = r.dhyp**2 - H**2
    if R2 <= 0: 
        print(f'  {r.sta}: lingkaran hilang (dhyp {r.dhyp:.1f} km < H {H} km)')
        continue
    ax.add_patch(Circle((sta.loc[r.sta,'lon'], sta.loc[r.sta,'lat']), np.sqrt(R2)/KM_PER_DEG,
                        fill=False, ec='#0e7490', lw=1))
ax.plot(sta.lon, sta.lat, '^', ms=9, color='#334155')
ax.plot(kat.loc[EVENT,'longitude'], kat.loc[EVENT,'latitude'], '*', ms=20, color='#dc2626', zorder=5)
ax.set_xlim(110.05,110.85); ax.set_ylim(-8.32,-7.58)
ax.set_aspect(1/np.cos(np.radians(7.9))); ax.set_xlabel('Bujur'); ax.set_ylabel('Lintang')
ax.set_title(f'{EVENT} — kedalaman dianggap {H} km')

✍️ **Jawaban 2:** Pada kedalaman berapa lingkarannya paling memusat? Bagaimana kalian tahu?

*(tulis di sini)*

## Bagian 3 — Inversi dengan pencarian kisi

Metode lingkaran tidak bisa memakai semua stasiun sekaligus dan tidak memberi galat.
Sekarang cari titik yang **meminimumkan RMS residu** atas seluruh stasiun.

Untuk medium seragam, waktu tempuh P dari titik uji ke stasiun adalah

$$T = \frac{\sqrt{\Delta x^2 + \Delta y^2 + z^2}}{V_P}$$

In [ ]:
def rms_residu(lat0, lon0, z0, dd, vp=VP):
    dx = (dd.lon.values - lon0) * KM_PER_DEG * np.cos(np.radians(lat0))
    dy = (dd.lat.values - lat0) * KM_PER_DEG
    Tc = np.sqrt(dx**2 + dy**2 + z0**2) / vp
    return np.sqrt(np.mean((dd.P.values - Tc)**2))

dd = d.join(sta, on='sta').dropna(subset=['lat','lon'])

lats  = np.arange(-8.15, -7.75, 0.005)
lons  = np.arange(110.20, 110.70, 0.005)
deps  = np.arange(1, 30, 1.0)

best = (1e9, None)
for z0 in deps:
    for la in lats:
        for lo in lons:
            r = rms_residu(la, lo, z0, dd)
            if r < best[0]: best = (r, (la, lo, z0))
rms, (LAT, LON, DEP) = best
ref = kat.loc[EVENT]
err = np.hypot((LON-ref.longitude)*KM_PER_DEG*np.cos(np.radians(LAT)), (LAT-ref.latitude)*KM_PER_DEG)
print(f'hasil kalian : {LAT:.4f}, {LON:.4f}, h = {DEP:.1f} km, RMS = {rms:.3f} s')
print(f'katalog      : {ref.latitude:.4f}, {ref.longitude:.4f}, h = {ref.depth:.1f} km')
print(f'galat episenter = {err:.2f} km   (ambang lulus 15 km)')

## Bagian 4 — Mengapa kedalaman berbeda sendiri

Buat irisan RMS: satu terhadap posisi mendatar, satu terhadap kedalaman.
Bandingkan lebar lembahnya.

In [ ]:
zz = np.arange(1, 30, 0.5)
rz = [rms_residu(LAT, LON, z, dd) for z in zz]
xx = np.arange(LON-0.15, LON+0.15, 0.002)
rx = [rms_residu(LAT, x, DEP, dd) for x in xx]

fig, ax = plt.subplots(1,2, figsize=(13,4))
ax[0].plot((xx-LON)*KM_PER_DEG*np.cos(np.radians(LAT)), rx); ax[0].set_xlabel('Geser mendatar (km)')
ax[1].plot(zz, rz); ax[1].set_xlabel('Kedalaman (km)')
for A in ax: A.set_ylabel('RMS residu (s)')
ax[0].set_title('Lembah terhadap posisi mendatar'); ax[1].set_title('Lembah terhadap kedalaman')

✍️ **Jawaban 4:** Lembah mana yang lebih lebar dan datar? Apa akibatnya bagi ketidakpastian
kedalaman dibandingkan ketidakpastian posisi mendatar? Kaitkan dengan *trade-off* kedalaman–waktu asal.

*(tulis di sini)*

## Bagian 5 — Bentuk lembah RMS: geometri baik vs buruk

Di kuliah kita lihat bahwa pada **populasi** 16.876 event, galat naik seiring gap. Tetapi pada
**satu event**, yang berubah bukan besarnya galat melainkan **bentuknya**.

Petakan permukaan RMS di sekitar solusi untuk satu event bergeometri baik (EV01–EV03) dan satu
bergeometri buruk (EV04–EV06). Ukur kelonjongan lembahnya.

In [ ]:
def permukaan_rms(ev, z=None, span=0.20, step=0.003):
    d  = tt[tt.event==ev].dropna(subset=['P','S']).join(sta, on='sta').dropna(subset=['lat','lon'])
    rf = kat.loc[ev]; z = rf.depth if z is None else z
    la = np.arange(rf.latitude-span,  rf.latitude+span,  step)
    lo = np.arange(rf.longitude-span, rf.longitude+span, step)
    R  = np.zeros((len(la), len(lo)))
    for i, a_ in enumerate(la):
        dx = (d.lon.values[None,:] - lo[:,None]) * KM_PER_DEG * np.cos(np.radians(a_))
        dy = (d.lat.values[None,:] - a_) * KM_PER_DEG
        R[i,:] = np.sqrt(np.mean((d.P.values[None,:] - np.sqrt(dx**2+dy**2+z**2)/VP)**2, axis=1))
    return la, lo, R, rf, d

def kelonjongan(la, lo, R, rf):
    m = R.min(); ii, jj = np.where(R < 1.1*m)
    y = (la[ii]-rf.latitude)*KM_PER_DEG
    x = (lo[jj]-rf.longitude)*KM_PER_DEG*np.cos(np.radians(rf.latitude))
    w, _ = np.linalg.eigh(np.cov(np.vstack([x, y])))
    return np.sqrt(max(w)/min(w))

fig, ax = plt.subplots(1, 2, figsize=(13, 5.6))
for j, ev in enumerate(['EV01', 'EV04']):        # <-- coba pasangan lain juga
    la, lo, R, rf, d = permukaan_rms(ev)
    X = (lo-rf.longitude)*KM_PER_DEG*np.cos(np.radians(rf.latitude)); Y = (la-rf.latitude)*KM_PER_DEG
    ax[j].contourf(X, Y, R, levels=20, cmap='YlGnBu_r')
    ax[j].contour(X, Y, R, levels=[1.1*R.min()], colors='red')
    for _, r in d.iterrows():                     # garis ke arah tiap stasiun
        ax[j].plot([0, (r.lon-rf.longitude)*KM_PER_DEG*np.cos(np.radians(rf.latitude))*3],
                   [0, (r.lat-rf.latitude)*KM_PER_DEG*3], '-', color='gray', lw=.6)
    ax[j].plot(0, 0, '*', ms=18, color='red')
    ax[j].set_xlim(-20, 20); ax[j].set_ylim(-20, 20); ax[j].set_aspect('equal')
    ax[j].set_title(f'{ev} — gap {rf.gap:.0f}°, kelonjongan {kelonjongan(la,lo,R,rf):.2f}')
    ax[j].set_xlabel('Timur (km)')
ax[0].set_ylabel('Utara (km)')

✍️ **Jawaban 5:**

1. Berapa kelonjongan lembah RMS untuk event bergeometri baik dan buruk?
2. **Ke arah mana** lembah pada event bergeometri buruk memanjang? Bandingkan dengan arah kumpulan stasiunnya.
3. Apakah RMS minimum event bergeometri buruk lebih besar atau lebih kecil? Mengapa jawabannya mungkin berlawanan dengan dugaan kalian — dan mengapa RMS kecil **bukan** berarti lokasinya bagus?

*(tulis di sini)*

## Bagian 6 — SEL BERGALAT ⚠️

Sel berikut ditulis AI dan tampak masuk akal. Ada **tiga kesalahan**. Temukan, perbaiki,
jelaskan akibatnya. Menuduh baris yang sebenarnya benar akan mengurangi nilai.

In [ ]:
# ============ SEL BERGALAT ============
# Menghitung jarak episentral dari selisih S-P lalu melokasikan gempa

dsp = d.SP.values

# (a) ubah S-P jadi jarak dengan kaidah lapangan
jarak = dsp * 8.0

# (b) karena yang dicari episenter, kedalaman diabaikan saja
R_peta = jarak

# (c) galat lokasi diambil dari RMS residu waktu, langsung sebagai kilometer
galat_km = rms

print('jarak (km) :', np.round(jarak, 1))
print('galat (km) :', round(galat_km, 3))
# ======================================

✍️ **Jawaban 6:**

| # | Baris | Kesalahannya | Akibatnya |
|:--|:--|:--|:--|
| 1 | | | |
| 2 | | | |
| 3 | | | |

## Bagian 7 — Setoran

**Kedalaman dinilai lewat kalibrasi, bukan ketepatan.** Laporkan selang, bukan satu angka.
Selang sempit yang meleset bernilai lebih rendah daripada selang lebar yang memuat nilai katalog —
karena pada titik ini kalian sudah tahu persis mengapa kedalaman sulit ditentukan.

In [ ]:
NIM = 'isi NIM kalian'          # <-- wajib, dipakai pengoreksi otomatis

setoran = dict(
    nim=NIM, nama=NAMA, soal=EVENT,
    lat=float(LAT), lon=float(LON), kedalaman_km=float(DEP),
    rms_s=float(rms), vpvs=float(vpvs), galat_episenter_km=float(err),
    kedalaman_selang_bawah=...,      # <-- selang keyakinan 80% kalian
    kedalaman_selang_atas=...,       # <-- JANGAN dikosongkan
)
pd.DataFrame([setoran]).to_csv(f'setoran_{NIM}_W12.csv', index=False)
setoran

✍️ **Catatan pemakaian AI** (wajib diisi):

- Apa yang kalian tanyakan?
- Bagian mana yang berasal dari sana?
- Bagian mana yang akhirnya kalian ubah sendiri, dan mengapa?

*(tulis di sini)*